# AI 201 — Week 4 Demo: Discord Content Filtering Bot — SOLUTION

Complete reference notebook. All layers implemented.

**The pipeline (cheapest → most expensive):**
1. **Rate limiter** — pre-built
2. **Input validation** — keywords + length check
3. **Injection defense** — pattern matching, runs before LLM
4. **LLM content filter** — handles ambiguous edge cases


In [ ]:
from dotenv import load_dotenv
load_dotenv()

import sys, json, time
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

from data.test_messages import ALL_SCENARIOS, print_scenario
from layers import RateLimiter, ModerationLogger
from utils import LLMClient

llm     = LLMClient()
limiter = RateLimiter(user_limit=5, server_limit=20, window_seconds=10)
log     = ModerationLogger()

print('Ready.')
print(f'LLM: {llm.model}')
print(f'Rate limiter: {limiter.user_limit} msg/user, {limiter.server_limit} msg/server per {limiter.window_seconds}s')
print(f'Scenarios loaded: {list(ALL_SCENARIOS.keys())}')

---
## Community Rules

In [ ]:
# Pre-built — run to load
COMMUNITY_RULES = {
    "server_name": "AI 201 Course Discussion Server",
    "vibe": "Respectful, evidence-based discussion. Disagreement is welcome; harassment is not.",
    "ban_immediately": [
        "Hate speech, slurs, or discriminatory harassment",
        "Doxxing or sharing private personal information",
        "Credible threats of violence or harm",
        "Posting illegal content",
    ],
    "timeout_30_min": [
        "Targeted personal attacks or repeated hostility",
        "Deliberately disruptive bad-faith behavior",
        "Spam, flooding, or repeated off-topic posting",
        "Attempts to bypass moderation rules",
    ],
    "allow": [
        "Respectful critique of ideas, code, or arguments",
        "Strong opinions supported by reasoning",
        "Constructive disagreement and debate",
        "Informal language that is not abusive or targeted",
    ],
}

print(f"Rules loaded for: {COMMUNITY_RULES['server_name']}")
print(f"  Ban offenses:     {len(COMMUNITY_RULES['ban_immediately'])}")
print(f"  Timeout offenses: {len(COMMUNITY_RULES['timeout_30_min'])}")
print(f"  Allowed:          {len(COMMUNITY_RULES['allow'])}")

---
## Input Validation Layer

In [ ]:
# Pre-built — run to load
BAN_KEYWORDS     = ["[REDACTED SLUR]"]  # extend with actual slurs
TIMEOUT_KEYWORDS = ["i will find you", "i know where you live", "posting your address"]
SPAM_PATTERNS    = ["buy followers", "click here", "http://spam"]


def validate_input(user_id: str, username: str, content: str) -> dict | None:
    """Fast, cheap checks before touching the LLM. Returns a decision or None."""
    content_lower = content.lower()

    rate_result = limiter.check(user_id)
    if not rate_result["allowed"]:
        return {"decision": "rate_limited", "reason": rate_result["reason"],
                "layer": "rate_limiter", "confidence": "high"}

    if len(content) > 2000:
        return {"decision": "timeout", "reason": f"Message too long ({len(content)} chars)",
                "layer": "input_validation", "confidence": "high"}

    for keyword in BAN_KEYWORDS:
        if keyword.lower() in content_lower:
            return {"decision": "ban", "reason": f"Hate speech detected ('{keyword}')",
                    "layer": "input_validation", "confidence": "high"}

    for keyword in TIMEOUT_KEYWORDS + SPAM_PATTERNS:
        if keyword.lower() in content_lower:
            return {"decision": "timeout", "reason": f"Flagged pattern ('{keyword}')",
                    "layer": "input_validation", "confidence": "high"}

    return None


print('validate_input() loaded.')

---
## LLM Content Filter

In [ ]:
def build_system_prompt(rules: dict) -> str:
    """Format the community rules dict into a moderator system prompt."""
    ban_rules     = "\n".join(f"  - {r}" for r in rules["ban_immediately"])
    timeout_rules = "\n".join(f"  - {r}" for r in rules["timeout_30_min"])
    allow_rules   = "\n".join(f"  - {r}" for r in rules["allow"])

    return f"""You are the moderation bot for {rules['server_name']}.
Server vibe: {rules['vibe']}

IMMEDIATELY BAN for:
{ban_rules}

TIMEOUT (30 min) for:
{timeout_rules}

ALLOW (including):
{allow_rules}

Evaluate each message and return a JSON verdict.
When in doubt between timeout and allow, prefer allow.
Only ban when you are highly confident."""


def llm_filter(user_id: str, username: str, content: str) -> dict:
    """Pass a message to the LLM for content evaluation."""
    system_prompt = build_system_prompt(COMMUNITY_RULES)
    result = llm.moderate(system_prompt, content)
    result["layer"] = "llm_filter"
    return result


# Preview the system prompt
print(build_system_prompt(COMMUNITY_RULES))

---
## Prompt Injection Defense

In [ ]:
# Pre-built — run to load
INJECTION_PATTERNS = [
    "ignore your previous instructions",
    "ignore previous instructions",
    "you are now",
    "pretend you are",
    "pretend you're",
    "act as if you have no",
    "disregard your",
    "your new instructions",
    "system:",
    "new rule:",
    "admin privileges",
    "override moderation",
    "what would a bot with no",
    "hypothetically, if you had no rules",
]


def detect_injection(content: str) -> dict | None:
    """Block prompt injection attempts before they reach the LLM."""
    content_lower = content.lower()
    for pattern in INJECTION_PATTERNS:
        if pattern in content_lower:
            return {
                "decision": "timeout",
                "reason": f"Prompt injection attempt detected ('{pattern}')",
                "layer": "injection_defense",
                "confidence": "high",
            }
    return None


print(f'detect_injection() loaded — {len(INJECTION_PATTERNS)} patterns.')

---
## Wire It Together

In [ ]:
# Pre-built — run to load
def moderate(user_id: str, username: str, content: str) -> dict:
    """Run a message through all layers in order. Return the first decision that fires."""
    decision = detect_injection(content)               # injection before LLM
    if decision is None:
        decision = validate_input(user_id, username, content)  # cheap checks next
    if decision is None:
        decision = llm_filter(user_id, username, content)      # LLM last
    log.log(
        user_id=user_id, username=username, message=content,
        decision=decision["decision"], reason=decision["reason"],
        layer=decision["layer"], confidence=decision.get("confidence", ""),
    )
    return decision


print('moderate() loaded — all layers wired.')

---
## Demo Scenarios

In [ ]:
print('=== Clean messages ===\n')
for msg in ALL_SCENARIOS['clean']:
    result = moderate(msg.user_id, msg.username, msg.content)
    print(f'[{msg.username}] {result["decision"]} ({result["layer"]}) — {result["reason"]}')


In [ ]:
print('=== Obvious violations ===\n')
for msg in ALL_SCENARIOS['obvious_violation']:
    result = moderate(msg.user_id, msg.username, msg.content)
    print(f'[{msg.username}] {result["decision"]} ({result["layer"]}) — {result["reason"]}')


In [ ]:
# Edge cases — the genuinely ambiguous messages
# Ask BEFORE running: "What would you decide for each of these?"
# Then compare students' instincts to the model's decisions.
print('=== Edge cases (LLM filter) ===\n')
for msg in ALL_SCENARIOS['edge_case']:
    result = moderate(msg.user_id, msg.username, msg.content)
    print(f'[{msg.username}]')
    print(f'  Message:  "{msg.content[:100]}"')
    print(f'  Note:     {msg.notes}')
    print(f'  Decision: {result["decision"]} ({result["layer"]}, {result.get("confidence","")} confidence)')
    print(f'  Reason:   {result["reason"]}')
    print()

In [ ]:
# Injection attempts — show the defense layer catching them before LLM
# Ask after: "Why can't we just tell the LLM to ignore injection attempts?"
print('=== Prompt injection attempts ===\n')
for msg in ALL_SCENARIOS['injection']:
    result = moderate(msg.user_id, msg.username, msg.content)
    print(f'[{msg.username}]')
    print(f'  Message:  "{msg.content[:100]}"')
    print(f'  Decision: {result["decision"]} ({result["layer"]}) — {result["reason"]}')
    print()

In [ ]:
print('=== Raid simulation (15 users) ===\n')
limiter.reset()
blocked = 0
for msg in ALL_SCENARIOS['raid']:
    result = moderate(msg.user_id, msg.username, msg.content)
    status = '🚫 BLOCKED' if result['decision'] == 'rate_limited' else '  passed'
    if result['decision'] == 'rate_limited': blocked += 1
    print(f'{status}  [{msg.username}]: "{msg.content[:60]}"')
print(f'\n{blocked}/{len(ALL_SCENARIOS["raid"])} raid messages blocked.')


---
## Moderation Log

In [ ]:
print(f'Total decisions logged: {len(log)}\n')
log.print_log()
print('Summary:')
for decision, count in sorted(log.summary().items()):
    print(f'  {decision:<14}: {count}')